In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

In [15]:
from selenium import webdriver
import time

# Set up WebDriver (Chrome)
driver = webdriver.Chrome()  # or use ChromeDriverManager for auto install

# Open URL
driver.get("https://www.cars24.com/buy-used-tata-punch-2022-cars-hyderabad-10417840780/")

# Wait for content to load
time.sleep(5)  # or use WebDriverWait for better control

# Get the fully rendered page source
html = driver.page_source

# Save to file
with open("page.html", "w", encoding="utf-8") as f:
    f.write(html)
# Close browser
driver.quit()


In [16]:
from bs4 import BeautifulSoup
import json
import pandas as pd
import os

def extract_car_details(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    car_details = {}

    # --- From JSON-LD Schema ---
    car_schema_script = soup.find('script', {'type': 'application/ld+json', 'id': 'carSchema'})
    if car_schema_script:
        try:
            schema_data = json.loads(car_schema_script.string)
            if schema_data.get('@type') == 'Car':
                car_details['Car make'] = schema_data.get('brand', {}).get('name')
                car_details['model'] = schema_data.get('model')

                mileage_data = schema_data.get('mileageFromOdometer', {})
                if mileage_data.get('unitCode') == 'KMT':
                    car_details['kilometers driven'] = f"{mileage_data.get('value', '')} km"
                else:
                    car_details['kilometers driven'] = None

                car_details['fuel type'] = schema_data.get('fuelType')

                offer_data = schema_data.get('offers', {})
                if offer_data:
                    price = offer_data.get('price')
                    currency = offer_data.get('priceCurrency')
                    car_details['price'] = f"{currency} {price}" if price and currency else None
                else:
                    car_details['price'] = None

                location_data = schema_data.get('location', {}).get('address', {})
                car_details['location'] = location_data.get('addressLocality') if location_data else None

        except json.JSONDecodeError:
            print("Error decoding JSON from carSchema script.")

    # --- Fallback from visible content ---
    car_name_tag = soup.find('h1', class_='sc-braxZu fjhfdl')
    if car_name_tag and not car_details.get('Car make'):
        full_name = car_name_tag.get_text(strip=True)
        parts = full_name.split()
        if len(parts) >= 2:
            car_details['Car make'] = parts[1]
            car_details['model'] = ' '.join(parts[2:])
        else:
            car_details['Car make'] = full_name
            car_details['model'] = None

    overview_section = soup.find('div', id='CATALOG_CDP_KNOW_YOUR_CAR')
    if overview_section:
        items = overview_section.find_all('li')
        for item in items:
            label_tag = item.find('p', class_='sc-braxZu jjIUAi')
            value_tag = item.find('p', class_='sc-braxZu kjFjan')
            if label_tag and value_tag:
                label = label_tag.get_text(strip=True).replace('<!-- -->', '').strip()
                value = value_tag.get_text(strip=True)
                if 'KM driven' in label and not car_details.get('kilometers driven'):
                    car_details['kilometers driven'] = value
                elif 'Fuel' in label and not car_details.get('fuel type'):
                    car_details['fuel type'] = value
                elif 'Transmission' in label and not car_details.get('transmission type'):
                    car_details['transmission type'] = value

    price_tag = soup.find('p', class_='sc-braxZu hhzsvw')
    if price_tag and not car_details.get('price'):
        car_details['price'] = price_tag.get_text(strip=True)

    location_tag = soup.find('p', class_='sc-braxZu dughoY')
    if location_tag and 'Hyderabad' in location_tag.get_text() and not car_details.get('location'):
        car_details['location'] = location_tag.get_text(strip=True)

    return car_details

# ----------- Loop through all HTML files and extract data ------------

car_data_list = []

# Update this to match your files (e.g., "page1.html" to "page10.html")
for i in range(1, 11):
    file_name = f"page{i}.html"
    if os.path.exists(file_name):
        with open(file_name, "r", encoding="utf-8") as f:
            html_content = f.read()
            data = extract_car_details(html_content)
            if data:
                car_data_list.append(data)
            else:
                print(f"No data extracted from {file_name}")
    else:
        print(f"{file_name} not found.")

# Convert list of dictionaries to DataFrame
df = pd.DataFrame(car_data_list)

# Display or save
print("\nExtracted Car DataFrame:")
print(df)

# Optional: Save to Excel or CSV
df.to_csv("car_data.csv", index=False)
# df.to_excel("car_data.xlsx", index=False)


page1.html not found.

Extracted Car DataFrame:
  Car make        model kilometers driven fuel type        price  \
0  Hyundai        Verna          54054 km    Petrol   INR 535383   
1  Hyundai        VENUE          72098 km    Petrol   INR 754586   
2  Hyundai      ALCAZAR          39681 km    Diesel  INR 1864868   
3   Maruti        Swift          36461 km    Petrol   INR 608581   
4   Maruti  New Wagon-R          35376 km    Petrol   INR 494011   
5   Maruti       Baleno          53135 km    Petrol   INR 495000   
6     Tata        NEXON          39337 km    Diesel  INR 1065650   
7     Tata       ALTROZ          24075 km    Petrol   INR 785764   
8     Tata        PUNCH          26639 km    Petrol   INR 721000   

                         location transmission type  
0  Upperpally, Attapur, Hyderabad            Manual  
1             Kompally, Hyderabad            Manual  
2           Bachupally, Hyderabad         Automatic  
3           Bachupally, Hyderabad            Manual  
4

In [17]:
df

,Car make,model,kilometers driven,fuel type,price,location,transmission type
0,Hyundai,Verna,54054 km,Petrol,INR 535383,"Upperpally, Attapur, Hyderabad",Manual
1,Hyundai,VENUE,72098 km,Petrol,INR 754586,"Kompally, Hyderabad",Manual
2,Hyundai,ALCAZAR,39681 km,Diesel,INR 1864868,"Bachupally, Hyderabad",Automatic
3,Maruti,Swift,36461 km,Petrol,INR 608581,"Bachupally, Hyderabad",Manual
4,Maruti,New Wagon-R,35376 km,Petrol,INR 494011,"Kompally, Hyderabad",Automatic
5,Maruti,Baleno,53135 km,Petrol,INR 495000,"Kompally, Hyderabad",Automatic
6,Tata,NEXON,39337 km,Diesel,INR 1065650,"Kompally, Hyderabad",Automatic
7,Tata,ALTROZ,24075 km,Petrol,INR 785764,"Kompally, Hyderabad",Manual
8,Tata,PUNCH,26639 km,Petrol,INR 721000,"Bachupally, Hyderabad",Manual
